In [ ]:
# ===============================
# 📦 INSTALL (run once)
# ===============================
!pip install groq langchain==0.1.16 pypdf gradio

# ===============================
# 🔑 IMPORTS
# ===============================
from groq import Groq
import gradio as gr
import re

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ===============================
# 🔐 API KEY
# ===============================
GROQ_API_KEY = ""
client = Groq(api_key=GROQ_API_KEY)

# ===============================
# 🔹 STEP 1: RESUME ANALYSIS
# ===============================
def analyze_resume(file, job_description):
    try:
        if file is None:
            return "⚠️ Please upload a resume.", [], 0

        loader = PyPDFLoader(file.name)
        docs = loader.load()

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=300,
            chunk_overlap=30
        )
        chunks = splitter.split_documents(docs)

        context = " ".join([doc.page_content for doc in chunks[:3]])

        prompt = f"""
You are a senior AI recruiter.
Use emojis, bold text, and formatting.

### Resume:
{context}

### Job Description:
{job_description}

### Output:

1. Match Score:
XX/100

2. Key Strengths:
- ...

3. Missing Skills:
- ...

4. Final Decision:
- PASS (if score ≥ 60)
- FAIL (if score < 60)

5. Message:
If PASS → Congratulate
If FAIL → Give 2-3 improvements + polite rejection
"""

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        result = response.choices[0].message.content

        # Extract score
        match = re.search(r'(\d+)/100', result)
        score = int(match.group(1)) if match else 0

        questions = []

        # ===============================
        # 🔹 STEP 2: QUESTION GENERATION
        # ===============================
        if score >= 60:
            question_prompt = f"""
Generate 2 EASY interview questions.

Resume:
{context}

Job:
{job_description}

Output:
Q1:
Q2:
"""

            q_response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": question_prompt}]
            )

            q_text = q_response.choices[0].message.content
            questions = re.findall(r'Q\d:\s*(.*)', q_text)

        return result, questions, score

    except Exception as e:
        return f"❌ Error: {str(e)}", [], 0


# ===============================
# 🔹 STEP 3: ANSWER EVALUATION
# ===============================
def evaluate_answers(questions, answers):
    try:
        prompt = f"""
Evaluate answers.

Questions:
{questions}

Answers:
{answers}

Output:
Q1 Score: X/10
Q2 Score: X/10
Average: X/10

Decision:
PASS ≥ 7
FAIL < 7

Message:
"""

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        return response.choices[0].message.content

    except Exception as e:
        return f"❌ Error: {str(e)}"


# ===============================
# 🎨 UI DESIGN
# ===============================
with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="blue", secondary_hue="violet")
) as demo:

    # 🔥 HEADER
    gr.Markdown("""
# 💼 Smart Recruitment AI
### 🚀 Resume Screening + AI Interview System
---
""")

    # 🧠 STATE
    state_questions = gr.State([])
    state_score = gr.State(0)

    # =============================
    # 📄 ANALYSIS SECTION
    # =============================
    with gr.Group():
        gr.Markdown("## 📄 Resume Analysis")

        with gr.Row():
            with gr.Column():
                file_input = gr.File(label="📄 Upload Resume (PDF)")

                job_input = gr.Textbox(
                    label="💼 Job Description",
                    placeholder="Enter job requirements...",
                    lines=6
                )

                analyze_btn = gr.Button("🚀 Analyze Candidate", variant="primary")

            with gr.Column():
                output = gr.Markdown(label="📊 Result")

    # =============================
    # 🎤 INTERVIEW SECTION
    # =============================
    with gr.Group():
        gr.Markdown("## 🎤 Interview Round")

        start_btn = gr.Button("🎯 Start Interview", visible=False)

        questions_box = gr.Markdown(visible=False)

        answers_box = gr.Textbox(
            label="✍️ Your Answers",
            lines=6,
            visible=False
        )

        evaluate_btn = gr.Button("📊 Evaluate", visible=False)

        final_output = gr.Markdown(visible=False)

    # =============================
    # 🔹 ANALYZE FUNCTION
    # =============================
    def handle_analysis(file, job):
        result, questions, score = analyze_resume(file, job)

        if score >= 60:
            result = "✅ " + result
            return result, questions, score, gr.update(visible=True)
        else:
            result = "❌ " + result
            return result, [], score, gr.update(visible=False)

    analyze_btn.click(
        fn=handle_analysis,
        inputs=[file_input, job_input],
        outputs=[output, state_questions, state_score, start_btn],
        show_progress=True
    )

    # =============================
    # 🔹 START INTERVIEW (FIXED)
    # =============================
    def start_interview(questions):
        formatted_q = "\n".join(
            [f"👉 **Q{i+1}:** {q}" for i, q in enumerate(questions)]
        )

        return (
            gr.update(value=formatted_q, visible=True),
            gr.update(visible=True),
            gr.update(visible=True)
        )

    start_btn.click(
        fn=start_interview,
        inputs=[state_questions],
        outputs=[questions_box, answers_box, evaluate_btn]
    )

    # =============================
    # 🔹 EVALUATE
    # =============================
    def handle_evaluation(questions, answers):
        result = evaluate_answers(questions, answers)
        return result, gr.update(visible=True)

    evaluate_btn.click(
        fn=handle_evaluation,
        inputs=[state_questions, answers_box],
        outputs=[final_output, final_output]
    )

    # FOOTER
    gr.Markdown("---")
    gr.Markdown("⚡ Built with Groq + LangChain + Gradio")

# ===============================
# 🚀 LAUNCH
# ===============================
demo.launch(share=True)

/tmp/ipykernel_764/2924956752.py:156: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dc86d603ef8b944e18.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
